# 09 — La regla de citación correcta: qué mide y qué alternativas admite

**Autor:** Jason Sepúlveda S.
**Proyecto:** Evaluación experimental de la extracción automática y la representación
persistente de decisiones organizacionales frente a Recuperación Aumentada (RAG)
**Depende de:** [`07-jss-runner-experimento-b.ipynb`](07-jss-runner-experimento-b.ipynb) (corrida de
humo con C1) · [ADR 0005](../docs/decisions/0005-adr-development-evaluation-split.md)

---

> **Estado: análisis descriptivo sobre datos de desarrollo.** Se usan las 21 preguntas de
> calibración de ES2015 y los registros de la corrida de humo con C1, de los cuales **13
> respuestas** son contestaciones a preguntas con evidencia de referencia. ES2015 es una
> **serie de desarrollo** (ADR 0005): la muestra es pequeña, el análisis es descriptivo y
> **ninguna cifra de este notebook es un resultado de la tesis**. No se corrió ningún modelo
> ni ningún experimento, y la regla vigente (`src/afg/evaluation/citation.py`) **no se
> modificó**. El objetivo es dar a la dirección del proyecto los elementos para decidir si la
> regla se mantiene o se cambia antes del Experimento B real; el notebook no elige.

# 1. Un caso concreto: la respuesta es correcta y cita el pasaje correcto, pero la citación se da por incorrecta

La pregunta q007 de ES2015 es: *¿qué elementos de la marca de la empresa debían incorporarse al
control remoto?* La respuesta de referencia es «el logotipo y el color corporativos». C1
responde lo mismo, y el juez automático la da por correcta.

Tres términos que se usan en todo el notebook:

- **Citación**: la lista de fragmentos de la transcripción que se presentan como respaldo de una
  respuesta. Aquí cada fragmento (en inglés, *chunk*) es un trozo de la reunión de unas decenas
  de intervenciones.
- **Acto de diálogo**: la unidad mínima en que el corpus AMI divide lo que dice cada persona
  (una frase o una intervención corta). Cada acto tiene un identificador; un fragmento contiene
  de decenas a más de cien actos.
- **Evidencia de referencia**: los actos de diálogo que la persona que preparó la pregunta
  señaló como el lugar donde está la respuesta. Es lo que la regla compara con lo citado.

La regla vigente dice que una citación es correcta **solo si cada uno de los fragmentos citados
comparte al menos un acto con la evidencia de referencia**. Las celdas siguientes muestran el
caso.

In [1]:
from dataclasses import asdict

import pandas as pd

from afg.evaluation.passage import evidence_passage
from afg.experiments.citation_analysis import (
    cited_chunk_summaries,
    compare_records,
    reference_sizes,
    tally_by_verdict,
    tally_variants,
)
from afg.experiments.questions import load_experiment_questions
from afg.experiments.runner import RECORDS_FILE, load_records
from afg.shared.paths import EXPERIMENT_B_DIR, PROJECT_ROOT

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 110)

RUN_DIR = EXPERIMENT_B_DIR / "humo-c1-es2015"
QUESTIONS_CSV = PROJECT_ROOT / "data" / "processed" / "judge-calibration" / "ES2015.preguntas.csv"
for needed in (RUN_DIR / RECORDS_FILE, QUESTIONS_CSV):
    if not needed.exists():
        raise SystemExit(
            f"No se encontro {needed}.\n"
            "Se genera con `uv run afg experiment b --help` (corrida de humo, requiere Ollama) y "
            "con la hoja de calibracion de ES2015."
        )

QUESTIONS = load_experiment_questions(QUESTIONS_CSV)
BY_ID = {q.id: q for q in QUESTIONS}
RECORDS = load_records(RUN_DIR / RECORDS_FILE, latest_only=True)
BY_RECORD = {r["question_id"]: r for r in RECORDS}
ROWS = compare_records(RECORDS, BY_ID)
print(
    f"{len(QUESTIONS)} preguntas · {len(RECORDS)} registros · "
    f"{len(ROWS)} respuestas con evidencia de referencia"
)

21 preguntas · 21 registros · 13 respuestas con evidencia de referencia


In [2]:
def show_case(question_id):
    question = BY_ID[question_id]
    record = BY_RECORD[question_id]
    print("Pregunta  :", question.text)
    print("Referencia:", question.reference_answer)
    print("Respuesta :", record["answer"]["text"].split("USED:")[0].strip())
    print(
        "Juez      : correcta =",
        record["verdict"]["correct"],
        "| respaldada =",
        record["verdict"]["evidence_supported"],
        "| citacion correcta (regla vigente) =",
        record["citation_correct"],
    )
    print("\nTexto de la evidencia de referencia (lo que dijo la persona en esos actos):")
    print(evidence_passage(question.reference_evidence, context_turns=0))
    print("\nFragmentos citados:")
    chunks = pd.DataFrame(
        [asdict(c) for c in cited_chunk_summaries(record, question.reference_evidence)]
    )
    return chunks.rename(
        columns={
            "chunk_id": "fragmento",
            "meeting_id": "reunion",
            "acts": "actos",
            "acts_in_reference": "actos en la referencia",
            "hit": "toca la referencia",
            "snippet": "comienzo del texto",
        }
    )


show_case("ES2015.q007")

Pregunta  : Which company branding elements had to be incorporated into the remote control?
Referencia: The corporate logo and colour.
Respuesta : The company branding elements that had to be incorporated into the remote control included the company's logo and colour. These elements were to be included in some way to ensure recognisability of the product. The logo could be placed on the bottom or wherever it would look good, and the colour was to be used to reinforce the brand identity. The remote control was expected to reflect the company's commitment to fashion in electronics and align with trends in clothing and interior design.
Juez      : correcta = True | respaldada = False | citacion correcta (regla vigente) = False

Texto de la evidencia de referencia (lo que dijo la persona en esos actos):
[ES2015b]
PM: more of a sense that people know that this is from our company. So, um, all the remote controls must have our um We'll incorporate our logo and colour in in some way. So, um, 

,fragmento,reunion,actos,actos en la referencia,toca la referencia,comienzo del texto
0,ES2015b.c029,ES2015b,25,1,True,"ME: Hmm. ID: Were they, was the management suggesting use of the internet rather than teletext, or just av..."
1,ES2015d.c020,ES2015d,43,0,False,"ID: Yeah, that's what I was thinking. We may have to do some more research into other strain injuries that..."


La respuesta cita dos fragmentos. El primero (de la reunión ES2015b) contiene el acto de
referencia: allí se pide incorporar el logotipo y el color. El segundo (ES2015d) es un repaso
posterior en que el equipo confirma que «sí lo incorporamos»: dice lo mismo, pero no es el acto
señalado como referencia: allí se lee, por ejemplo, «¿incorporamos de alguna manera el color y
el logotipo de la empresa? […] Sí, lo hicimos». Como la regla exige que **todos** los
fragmentos citados toquen la referencia, la citación se da por incorrecta, aunque ambos
fragmentos respaldan el contenido central de la respuesta (el logotipo y el color).
(La tabla muestra solo el comienzo de cada fragmento, que no es la parte pertinente.)

La pregunta de este notebook es si esa exigencia es la que se quiere medir.

---

# 2. Qué dice la tesis que mide la métrica

La propuesta define la métrica en §5.6 (Métricas, Respuesta OE3):

> «**tasa de citación correcta** (la fuente citada respalda efectivamente la afirmación —
> distinta de la anterior, y la que detecta el modo de falla más peligroso: respuesta correcta
> con cita que no la sustenta)»

y §6.2 (Transparencia y explicabilidad) fija el criterio de trazabilidad:

> «toda decisión extraída mantiene trazabilidad hacia el fragmento de transcripción que la
> respalda […] Una afirmación sin fuente identificable cuenta como falla aunque sea correcta.»

Dos consecuencias para leer las variantes. Primero, el texto habla de «la fuente citada»
(en singular) que «respalda efectivamente la afirmación»: no precisa qué ocurre cuando se citan
varias fuentes y solo algunas respaldan. Segundo, el modo de falla que se quiere detectar es una
**cita que no sustenta la respuesta**; una respuesta sin ninguna cita se considera falla (§6.2).
La regla vigente concreta «respalda» como «comparte al menos un acto con la evidencia de
referencia», y «las citas» como «todas las citas». Ambas son decisiones de implementación, no
palabras de la tesis. La metodología (`docs/avances/06-jss-metodologia.md`) solo enumera la
métrica («citación correcta») sin definirla más.

---

# 3. Cuánta evidencia de referencia tiene cada pregunta

Primero, qué tan dispersa es la referencia. Cada fila es una pregunta con respuesta (las de
estrato E4, sin respuesta en la reunión, no tienen evidencia de referencia y quedan fuera).
«Actos» es el número de actos de diálogo señalados como evidencia; «reuniones», en cuántas
reuniones distintas están.

In [3]:
SIZES = pd.DataFrame([asdict(s) for s in reference_sizes(QUESTIONS)]).rename(
    columns={"question": "pregunta", "stratum": "estrato", "acts": "actos", "meetings": "reuniones"}
)
SIZES.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
pregunta,q002,q003,q004,q006,q007,q008,q010,q011,q013,q016,q017,q020,q023,q026,q028,q029,q030
estrato,E1,E1,E1,E1,E1,E1,E1,E1,E1,E1,E1,E1,E1,E1,E3,E2,E2
actos,1,1,1,1,1,1,1,2,2,1,1,3,3,1,4,2,3
reuniones,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2


In [4]:
print("Preguntas con evidencia de referencia:", len(SIZES))
for column in ("actos", "reuniones"):
    counts = SIZES[column].value_counts().sort_index().rename_axis(column).to_frame("preguntas")
    print(f"\nDistribucion del numero de {column} de referencia:")
    print(counts.T.to_string())
print(f"\nMediana: {SIZES['actos'].median():.0f} acto(s); maximo: {SIZES['actos'].max()}")

Preguntas con evidencia de referencia: 17

Distribucion del numero de actos de referencia:
actos       1  2  3  4
preguntas  10  3  3  1

Distribucion del numero de reuniones de referencia:
reuniones   1  2
preguntas  14  3

Mediana: 1 acto(s); maximo: 4


La referencia es **muy escasa**: 10 de las 17 preguntas tienen un único acto de diálogo como
evidencia (una frase o una intervención corta), 6 tienen dos o tres y ninguna pasa de cuatro.
Las tres preguntas cuya referencia abarca dos reuniones son las de estado vigente y evolución
(E2 y E3).

---

# 4. Cuánto cita cada respuesta

Del otro lado, qué tan grande es lo citado. C1 cita **fragmentos completos** de la reunión, y
cada fragmento se expande a todos sus actos. «Fragmentos citados» es la cantidad de piezas de
la citación; «actos citados» es el total de actos distintos que contienen.

In [5]:
TABLE = pd.DataFrame([asdict(r) for r in ROWS])
sizes = TABLE[["question", "cited_spans", "cited_acts", "reference_acts"]].rename(
    columns={
        "question": "pregunta",
        "cited_spans": "fragmentos citados",
        "cited_acts": "actos citados",
        "reference_acts": "actos de referencia",
    }
)
sizes

,pregunta,fragmentos citados,actos citados,actos de referencia
0,q002,3,149,1
1,q003,3,117,1
2,q004,1,24,1
3,q006,1,65,1
4,q007,2,68,1
5,q010,2,59,1
6,q011,2,52,2
7,q013,1,62,2
8,q017,2,67,1
9,q020,2,121,3


In [6]:
print("Resumen sobre las", len(ROWS), "respuestas:")
sizes[["fragmentos citados", "actos citados", "actos de referencia"]].describe().loc[
    ["min", "50%", "mean", "max"]
].round(1)

Resumen sobre las 13 respuestas:


,fragmentos citados,actos citados,actos de referencia
min,1.0,24.0,1.0
50%,2.0,68.0,1.0
mean,1.8,80.8,1.7
max,3.0,149.0,3.0


Se cita del orden de **decenas a más de cien actos** para respaldar una evidencia de referencia
de **uno a tres actos**. Esa diferencia de escala es el núcleo del problema: la referencia es
un acto puntual y la cita es una región de la reunión. Que un fragmento «toque» la referencia
es una condición exigente (hay que citar justo el fragmento que contiene el acto señalado), y
exigir que todos los fragmentos lo toquen la vuelve más exigente aún.

---

# 5. Variantes de la regla, lado a lado

Un fragmento citado **toca** la referencia cuando comparte reunión y al menos un acto con ella.
Sobre esa base se comparan cuatro lecturas de «la fuente citada respalda la afirmación», cada
una tomada literalmente de su definición:

| Variante | Definición | Qué mediría |
|---|---|---|
| **(a) Todos tocan** (regla vigente) | cada fragmento citado toca la referencia | que no haya ninguna cita «de relleno» |
| **(b) Alguno toca** | al menos un fragmento citado toca la referencia | que la respuesta cite, entre otras cosas, la fuente correcta |
| **(c) Precisión de la citación** | fracción de fragmentos citados que tocan la referencia | cuánto de lo citado es pertinente (valor entre 0 y 1) |
| **(d) Cobertura de la referencia** | fracción de los actos de referencia incluidos en lo citado | si la citación contiene toda la evidencia señalada (valor entre 0 y 1) |

Una respuesta sin ninguna cita falla las cuatro variantes (§6.2). Las variantes (a) y (b) son
de sí o no; (c) y (d) son proporciones. La tabla muestra las 13 respuestas con las cuatro
lecturas y el veredicto del juez. «USED en línea» marca las dos respuestas cuya atribución se
perdió por el marcador `USED:` escrito al final de una oración (`jasonssdev/openkos#1384`,
`jasonssdev/mcd-afg#87`): en ellas se citaron todos los fragmentos recuperados y no solo los
que el modelo declaró haber usado.

In [7]:
COMPARISON = TABLE.rename(
    columns={
        "question": "pregunta",
        "correct": "correcta",
        "evidence_supported": "respaldada",
        "inline_used_marker": "USED en línea",
        "cited_spans": "fragmentos",
        "all_hit": "(a) todos tocan",
        "any_hit": "(b) alguno toca",
        "precision": "(c) precisión",
        "recall": "(d) cobertura",
    }
)[
    [
        "pregunta",
        "correcta",
        "respaldada",
        "USED en línea",
        "fragmentos",
        "(a) todos tocan",
        "(b) alguno toca",
        "(c) precisión",
        "(d) cobertura",
    ]
]
COMPARISON.round(2)

,pregunta,correcta,respaldada,USED en línea,fragmentos,(a) todos tocan,(b) alguno toca,(c) precisión,(d) cobertura
0,q002,True,True,True,3,False,True,0.33,1.00
1,q003,True,False,False,3,False,True,0.33,1.00
2,q004,True,True,False,1,True,True,1.00,1.00
3,q006,True,True,False,1,True,True,1.00,1.00
4,q007,True,False,False,2,False,True,0.50,1.00
5,q010,True,True,False,2,False,False,0.00,0.00
6,q011,False,False,False,2,False,False,0.00,0.00
7,q013,True,False,False,1,False,False,0.00,0.00
8,q017,True,True,False,2,False,False,0.00,0.00
9,q020,True,True,True,2,False,True,0.50,0.33


---

# 6. Cuántas respuestas correctas castiga cada variante

El riesgo que la tesis quiere detectar es la respuesta **correcta con una cita que no la
sustenta**. El riesgo inverso, que preocupa al leer la tabla anterior, es que la regla castigue
respuestas correctas cuya cita sí las respalda. La tabla cuenta, para cada variante, cuántas de
las respuestas **correctas** (según el juez) quedan marcadas con citación incorrecta, primero
con las 13 respuestas y luego separando las dos con `USED:` en línea.

In [8]:
def tally_table(rows):
    t = tally_variants(rows)
    return {
        "respuestas": t.records,
        "correctas (juez)": t.correct_answers,
        "(a) todos tocan: cumplen": t.all_hit,
        "(b) alguno toca: cumplen": t.any_hit,
        "correctas castigadas por (a)": t.correct_but_all_miss,
        "correctas castigadas por (b)": t.correct_but_any_miss,
        "(c) precisión media": round(t.mean_precision, 2),
        "(d) cobertura media": round(t.mean_recall, 2),
    }


GROUPS = {
    "todas": ROWS,
    "sin las 2 con USED en línea": [r for r in ROWS if not r.inline_used_marker],
    "solo las 2 con USED en línea": [r for r in ROWS if r.inline_used_marker],
}
pd.DataFrame({name: tally_table(rows) for name, rows in GROUPS.items()})

,todas,sin las 2 con USED en línea,solo las 2 con USED en línea
respuestas,13.00,11.00,2.00
correctas (juez),12.00,10.00,2.00
(a) todos tocan: cumplen,5.00,5.00,0.00
(b) alguno toca: cumplen,9.00,7.00,2.00
correctas castigadas por (a),7.00,5.00,2.00
correctas castigadas por (b),3.00,3.00,0.00
(c) precisión media,0.51,0.53,0.42
(d) cobertura media,0.59,0.58,0.67


Lectura. La regla vigente (a) da por incorrecta la citación de 8 de las 13 respuestas, y 7 de
esas 8 son correctas según el juez. La variante (b) rechaza 4 de las 13 y castiga a 3
respuestas correctas. Las dos respuestas con `USED:` en línea fallan (a) porque se citaron
todos los fragmentos recuperados, pero cumplen (b): no explican la mayor parte de la
severidad, que persiste en las otras 11 (6 de ellas incorrectas bajo (a)).

Cruce con el veredicto de **respaldo de evidencia** del juez (que sí lee el texto de la
transcripción, a diferencia de la regla de citación):

In [9]:
by_verdict = tally_by_verdict(ROWS)
pd.DataFrame(
    [
        {
            "correcta": c,
            "respaldada": s,
            "respuestas": t.records,
            "cumplen (a)": t.all_hit,
            "cumplen (b)": t.any_hit,
        }
        for (c, s), t in by_verdict.items()
    ]
)

,correcta,respaldada,respuestas,cumplen (a),cumplen (b)
0,True,True,7,3,5
1,True,False,5,2,4
2,False,False,1,0,0


Hay respuestas correctas y respaldadas por el juez cuya citación la regla vigente rechaza: el
juez ve respaldo en el texto y la regla de citación no lo ve en los actos señalados. Son dos
instrumentos que miden cosas distintas, y la diferencia es justamente la que §5.6 pide
distinguir. Queda por decidir si la regla de citación debe exigir el acto puntual de la
referencia o la pertinencia de la fuente.

---

# 7. Tres casos donde la regla vigente dice «incorrecta» y la respuesta es correcta

Los casos se leen con el texto de los actos de referencia (de la transcripción congelada de
ES2015) y un resumen de los fragmentos citados. Todos son de una serie de desarrollo, donde
leer contenido es admisible (ADR 0005).

### Caso A — q007 (la cita contiene la referencia y un repaso posterior)

Ya se vio en la sección 1: un fragmento toca la referencia y otro es un repaso posterior que
dice lo mismo. Falla (a); cumple (b).

### Caso B — q010: la cita habla del tema, pero no contiene el acto señalado

In [10]:
show_case("ES2015.q010")

Pregunta  : What feature did the team approve to help find a misplaced remote control?
Referencia: A locator alarm system that would make the remote emit a sound when activated.
Respuesta : The team approved an alarm system to help find a misplaced remote control. This feature would allow users to locate their remote control more easily when it goes missing.
Juez      : correcta = True | respaldada = True | citacion correcta (regla vigente) = False

Texto de la evidencia de referencia (lo que dijo la persona en esos actos):
[ES2015b]
PM: Yeah. I think we should definitely go ahead with the alarm system idea that you had. 'Cause I'm sure that could be inex inexpensive because we could use the same kind of infrared

Fragmentos citados:


,fragmento,reunion,actos,actos en la referencia,toca la referencia,comienzo del texto
0,ES2015a.c010,ES2015a,26,0,False,"ME: Um ID: I always find that in our house the remote control always goes missing. It's always, where is t..."
1,ES2015b.c020,ES2015b,33,0,False,"ID: Yeah. UI: Uh, probably just tuning in the channels, would it be? Tuning them in at the very start. You..."


La respuesta correcta es que el equipo aprobó un sistema de alarma para ubicar el control. Los
dos fragmentos citados mencionan la idea de la alarma (uno la propone: «algún tipo de alarma
[…] porque siempre se pierde»; otro la recuerda: «cómo podríamos tener un sistema de alarma
para que la gente lo encuentre»), pero
**ninguno contiene** el acto en que el proyecto la aprueba. Falla (a) y (b). Si la métrica
busca «la fuente respalda la afirmación», la cita es discutible: hay respaldo temático, no el
acto de la aprobación.

### Caso C — q017: el hecho está en otra parte de la reunión

In [11]:
show_case("ES2015.q017")

Pregunta  : What kind of buttons did the team choose?
Referencia: Push buttons.
Respuesta : The team considered various types of buttons, including pushbuttons and scroll buttons. They ultimately decided that pushbuttons would be the most straightforward option, though they also mentioned the possibility of incorporating scroll buttons with pushbuttons. However, the final choice leaned toward pushbuttons due to their simplicity and familiarity.
Juez      : correcta = True | respaldada = True | citacion correcta (regla vigente) = False

Texto de la evidencia de referencia (lo que dijo la persona en esos actos):
[ES2015c]
PM: I think we decided on the pushbuttons, right?

Fragmentos citados:


,fragmento,reunion,actos,actos en la referencia,toca la referencia,comienzo del texto
0,ES2015c.c005,ES2015c,28,0,False,PM: Yeah. ID: Um right our choice for buttons as well. We've developed some we've got some good advances i...
1,ES2015d.c011,ES2015d,39,0,False,"ID: I uh two, three. ME: Two. Okay, actu that's pro that's gonna get confusing, like that. Okay so questio..."


La referencia es la frase «creo que decidimos los pulsadores, ¿no?». Los dos fragmentos
citados hablan de pulsadores (uno dice «tenemos el pulsador integrado, que es el más
sencillo»; el otro, «es todo pulsadores»), de modo que la
respuesta «el equipo eligió pulsadores» tiene respaldo en el texto, pero ninguno incluye el
acto señalado. Falla (a) y (b). Casos como este muestran la dificultad de fondo: el mismo
hecho puede aparecer en varios lugares de la reunión y la referencia solo señala uno.

---

# 8. ¿Hay más casos en la calibración del juez?

Los elementos de `data/processed/judge-calibration/` (variantes *grounded* y *ungrounded*)
podrían sumar casos si traen una citación con la misma forma. La celda revisa sus columnas.

In [12]:
import csv

CAL = PROJECT_ROOT / "data" / "processed" / "judge-calibration" / "calibracion-juez.csv"
with CAL.open(newline="", encoding="utf-8-sig") as handle:
    header = next(csv.reader(handle))
print(header)
cited_columns = [c for c in header if "cit" in c.lower() and c != "citation_correct"]
print("\nColumnas con la evidencia citada por la respuesta:", cited_columns or "ninguna")

['id', 'series_id', 'stratum', 'question', 'reference_answer', 'reference_evidence', 'answer_text', 'pasaje_transcripcion', 'archivos_transcripcion', 'answer_abstained', 'correct', 'evidence_supported', 'citation_correct', 'labeler_a', 'labeler_b', 'adjudicated']

Columnas con la evidencia citada por la respuesta: ninguna


Los elementos de calibración traen la respuesta como texto (`answer_text`) y el pasaje de
transcripción que lee el juez, pero **no** una lista de actos citados por la respuesta
(`citation_correct` es una etiqueta de los anotadores, sin los actos citados detrás). No se pueden
evaluar con la regla ni con sus variantes, que necesitan los actos citados; por eso no se
agregan casos de ahí. Los 13 registros de la corrida de humo son la única muestra con citación
completa.

---

# 9. Qué no permite concluir este análisis

- Son **13 respuestas** de una sola serie de desarrollo y de una sola condición (C1), con un
  único modelo generador. Las proporciones son descriptivas y no estiman nada.
- Las cifras dependen del tamaño de los fragmentos de C1 (decenas de actos). C2 y C3 citarán
  de otra forma (actos o decisiones concretas), y el efecto de cada variante será distinto:
  una citación más puntual haría la regla vigente menos severa. La elección de la regla
  debería, por tanto, justificarse para las tres condiciones y no solo para C1.
- El tamaño de la referencia depende de la anotación. Con la anotación adjudicada de las
  series de evaluación podría cambiar.
- El veredicto del juez es automático (sin adjudicación humana de estas respuestas).

---

# 10. Opciones para la decisión

Qué haría cada opción con la métrica, en relación con §5.6 («la fuente citada respalda
efectivamente la afirmación») y §6.2 («una afirmación sin fuente identificable cuenta como
falla»). Las cifras son las de la sección 6 sobre las 13 respuestas de desarrollo, a modo de
orientación y no como estimación.

| Opción | A favor | En contra | Con las 13 respuestas |
|---|---|---|---|
| **Mantener (a)** todos los fragmentos deben tocar la referencia | Detecta cualquier cita de relleno o ajena; coincide con una lectura estricta de «la fuente respalda» para cada fuente citada; ya está implementada y probada | Penaliza citar más fragmentos aunque uno sea el correcto; castiga repasos válidos (caso A); depende de que la referencia señale el acto exacto; penaliza a las condiciones que citan más (C1 con atribución perdida) | 5 de 13 cumplen; 7 respuestas correctas castigadas |
| **Cambiar a (b)** al menos un fragmento toca la referencia | Acerca la métrica al modo de falla que describe §5.6 (cita que no sustenta la respuesta); no castiga citar de más | Admite citas de relleno mientras una sea correcta; no distingue citar un fragmento pertinente de citar diez; premia a quien cita más (más oportunidades de acertar) | 9 de 13 cumplen; 3 respuestas correctas castigadas |
| **Reportar (c)** precisión de la citación | Resume en un número cuánto de lo citado es pertinente; no es una decisión de sí o no; permite comparar condiciones con distinto volumen de citas | Se aleja de una «tasa» de respuestas con citación correcta tal como la nombra §5.6; hace falta fijar un umbral si se quiere una tasa; sensible al tamaño de los fragmentos | media 0,51 |
| **Complementar con (d)** cobertura de la referencia | Mide si la citación incluye la evidencia señalada; relevante para E2 y E3, con varios actos en varias reuniones | Con referencia de un solo acto casi siempre vale 0 o 1; no dice nada sobre citas sobrantes; castiga fragmentos grandes menos que pequeños | media 0,59 |
| **Combinar** (por ejemplo, tasa por (b) más precisión (c) como medida auxiliar) | Cubre el modo de falla central y deja ver el volumen de lo citado | Más de una cifra que explicar; hay que decir cuál es la métrica principal | — |

Ninguna opción corrige por sí sola la diferencia de escala entre una referencia de uno a tres
actos y una cita de decenas de actos. Una decisión relacionada, fuera de este análisis, es si
la referencia debe ampliarse (por ejemplo, con los actos que repiten el mismo hecho) o si la
citación de C1 debe ser más puntual; ambas cambiarían la severidad de cualquier variante.

La decisión corresponde a la dirección del proyecto. Cualquier cambio de regla debe quedar
registrado en `docs/decisions/` antes del Experimento B real, y no modifica las cifras de la
corrida de humo, que son de desarrollo.